# 12 · Derived Data — The Unbundled Database

**Part 1 · Data-Intensive Applications** · *Prerequisites: 02, 03, 11* · *Book mapping: DDIA ch. 12*

Every real application already runs a zoo: a primary database, a search index, caches, an
analytics store. Each holds *the same information in a different shape* — nb 02's models,
nb 03's engines, all at once. The question that decides whether the zoo thrives or rots:
**what keeps them consistent with each other?**

This finale builds the answer DDIA argues for: designate ONE **system of record** (an
append-only event log), and make everything else **derived data** — views maintained by
stream consumers (nb 11), rebuildable from the log at will (nb 10's batch superpower). A
database, unbundled: the log is the commit log, and the "storage engines" are subscribers.

**Learning objectives.**

1. Demonstrate why **dual writes** (application writes to N stores directly) cannot be made
   consistent — then route the same writes through a log and watch divergence become
   impossible.
2. Build the unbundled database: one log feeding a table, a search index, a denormalized
   profile cache, and analytics — each an independent consumer.
3. Use the two superpowers: **backfill a brand-new view** from history, and **fix a bug by
   rebuilding** a view (human fault tolerance).
4. Reason about the price: asynchrony — timeliness vs integrity, and cross-view reads.

In [ ]:
from collections import Counter, defaultdict

import matplotlib.pyplot as plt

from sysdes import viz

viz.use_style()

## 1. Failure first: the dual-write trap

The obvious architecture: on every update, the application writes the database, then the
cache, then the search index. Two problems, both fatal, both unfixable by care:

In [ ]:
db: dict[str, str] = {}
cache: dict[str, str] = {}

# Race: two clients update the same profile "simultaneously". Each writes both
# stores, but their messages interleave differently at each store (nb 08's
# network reordering - guaranteed to happen eventually):
db["bio"] = "I love databases"        # client 1 reaches the DB first...
db["bio"] = "I love caches"           # ...then client 2 overwrites: DB says 'caches'
cache["bio"] = "I love caches"        # client 2 reaches the cache first...
cache["bio"] = "I love databases"     # ...then client 1: cache says 'databases'

print(f"db: {db['bio']!r}   cache: {cache['bio']!r}")
assert db["bio"] != cache["bio"], \
    "DUAL-WRITE RACE: two stores, two opinions, FOREVER (no process will repair this)"

# And the crash variant: write one store, die before the other.
db["plan"] = "premium"
# -- crash -- (the cache write never happens)
assert "plan" not in cache, "dual-write crash: silent, permanent divergence"

The root cause is structural: with N independent stores, *there is no single order of
writes* — each store sees its own order (a mini version of nb 09's total-order problem),
and no store knows whether the others got the memo. The fix is to create the missing single
order: **write once, to a log; let every store follow the log.**

In [ ]:
log: list[dict] = []                                   # THE system of record
log.append({"type": "bio_set", "value": "I love databases"})   # client 1
log.append({"type": "bio_set", "value": "I love caches"})      # client 2
# The race still happened - but it was SETTLED, once, at append time.

def fold_bio(events):
    state = None
    for e in events:
        if e["type"] == "bio_set":
            state = e["value"]
    return state

db_view, cache_view = fold_bio(log), fold_bio(log)
assert db_view == cache_view == "I love caches", \
    "every consumer folds the SAME order -> disagreement is structurally impossible"
print(f"log-derived views agree: {db_view!r}")

## 2. Build: one log, four views

The application now emits **events** (facts, nb 11's event sourcing) to a single ordered
log. Around it we hang four derived views — deliberately spanning nb 02's data models and
nb 03's engines, because that's the point: *each consumer shapes the same truth for its own
queries*.

In [ ]:
fig, ax = viz.system_diagram(
    nodes=[
        {"id": "app", "x": 0, "y": 1.0, "label": "application\n(writes = events)", "kind": "client"},
        {"id": "log", "x": 2.8, "y": 1.0, "label": "event log\n(system of record)", "kind": "queue"},
        {"id": "tab", "x": 6.0, "y": 2.4, "label": "table view\n(current state)", "kind": "store"},
        {"id": "search", "x": 6.0, "y": 1.4, "label": "search index\n(inverted)", "kind": "store"},
        {"id": "cachev", "x": 6.0, "y": 0.4, "label": "profile cache\n(denormalized)", "kind": "cache"},
        {"id": "ana", "x": 6.0, "y": -0.6, "label": "analytics\n(counters)", "kind": "store"},
    ],
    edges=[
        {"src": "app", "dst": "log", "label": "append"},
        {"src": "log", "dst": "tab", "label": "offset 42"},
        {"src": "log", "dst": "search", "label": "offset 42"},
        {"src": "log", "dst": "cachev", "label": "offset 17 (lagging)"},
        {"src": "log", "dst": "ana", "label": "offset 42"},
    ],
    title="The unbundled database: one ordered truth, many derived shapes",
)
plt.show()

In [ ]:
class View:
    """A derived view = a fold over the log + a cursor (its offset).
    Subclasses implement apply(event). Determinism is the only requirement."""

    def __init__(self):
        self.offset = 0

    def process(self, log: list[dict], upto: int | None = None) -> None:
        upto = len(log) if upto is None else upto
        while self.offset < upto:
            self.apply(log[self.offset])
            self.offset += 1


class TableView(View):
    """Current state per entity - what a key-value/relational store would hold."""
    def __init__(self):
        super().__init__()
        self.users: dict[str, dict] = {}
        self.posts: dict[str, dict] = {}

    def apply(self, e):
        if e["type"] == "user_upserted":
            self.users[e["user_id"]] = {"name": e["name"]}
        elif e["type"] == "post_created":
            self.posts[e["post_id"]] = {"user_id": e["user_id"], "text": e["text"]}


class SearchView(View):
    """Full-text index: token -> set of post_ids (nb 03's OLAP cousin, nb 10's
    canonical batch product - here maintained incrementally)."""
    def __init__(self, lowercase: bool = True):
        super().__init__()
        self.lowercase = lowercase
        self.index: dict[str, set] = defaultdict(set)

    def apply(self, e):
        if e["type"] == "post_created":
            for token in e["text"].split():
                token = token.lower() if self.lowercase else token
                self.index[token].add(e["post_id"])

    def find(self, word: str) -> set:
        return self.index.get(word.lower() if self.lowercase else word, set())


class ProfileCacheView(View):
    """nb 02's denormalized document - but DERIVED, so its update anomaly is
    gone: nobody hand-repairs copies; the fold rebuilds them from the truth."""
    def __init__(self):
        super().__init__()
        self.pages: dict[str, dict] = {}

    def apply(self, e):
        if e["type"] == "user_upserted":
            page = self.pages.setdefault(e["user_id"], {"posts": [], "likes": 0})
            page["name"] = e["name"]
        elif e["type"] == "post_created":
            self.pages.setdefault(e["user_id"], {"posts": [], "likes": 0})
            self.pages[e["user_id"]]["posts"].append(e["text"])
        elif e["type"] == "post_liked":
            owner = e["owner_id"]
            self.pages.setdefault(owner, {"posts": [], "likes": 0})
            self.pages[owner]["likes"] += 1


class LikesView(View):
    """Analytics counters (the OLAP corner)."""
    def __init__(self):
        super().__init__()
        self.likes = Counter()

    def apply(self, e):
        if e["type"] == "post_liked":
            self.likes[e["post_id"]] += 1

# ---- The application writes ONLY to the log:
import numpy as np
rng = np.random.default_rng(seed=12)
log = []
for i in range(5):
    log.append({"type": "user_upserted", "user_id": f"u{i}", "name": f"User {i}"})
words = ["coffee", "systems", "logs", "cats", "streams", "design"]
posts = []
for j in range(12):
    owner = f"u{int(rng.integers(5))}"
    text = " ".join(rng.choice(words, size=4)) + (" Coffee" if j == 7 else "")
    posts.append((f"p{j}", owner))
    log.append({"type": "post_created", "post_id": f"p{j}", "user_id": owner,
                "text": text})
for _ in range(40):
    pid, owner = posts[int(rng.integers(len(posts)))]
    log.append({"type": "post_liked", "post_id": pid, "owner_id": owner})

table, search, cache_view, likes = TableView(), SearchView(), ProfileCacheView(), LikesView()
for v in (table, search, cache_view, likes):
    v.process(log)

# Cross-view consistency, verified against the log itself:
assert len(table.posts) == 12 and len(table.users) == 5
assert sum(likes.likes.values()) == 40, "analytics counted every like event exactly once"
for pid in search.find("coffee"):
    assert "coffee" in table.posts[pid]["text"].lower(), "search hits are real posts"
recomputed_pages = {u: [e["text"] for e in log
                        if e["type"] == "post_created" and e["user_id"] == u]
                    for u in table.users}
assert all(cache_view.pages[u]["posts"] == recomputed_pages[u] for u in table.users), \
    "the denormalized cache equals a from-scratch recomputation - no anomalies possible"
print("four views, one truth: all consistent with fold(log)")

## 3. The superpowers

### 3.1 A new view, with complete history

Six months in, product wants a posts-per-user leaderboard. In the dual-write world this
needs a risky migration script crawling the primary. Here: write the fold, start it at
offset 0. The log **is** the backfill:

In [ ]:
class LeaderboardView(View):
    def __init__(self):
        super().__init__()
        self.counts = Counter()
    def apply(self, e):
        if e["type"] == "post_created":
            self.counts[e["user_id"]] += 1

board = LeaderboardView()
board.process(log)                     # replays ALL history, then stays current
truth = Counter(e["user_id"] for e in log if e["type"] == "post_created")
assert board.counts == truth, "born yesterday, yet knows everything"
print(f"new view backfilled from history: {board.counts.most_common(3)}")

### 3.2 Fixing a bug by rebuilding (human fault tolerance)

A search view shipped without lowercasing. Users can't find the "Coffee" post by searching
`coffee`. In a mutate-in-place index this is a data-repair project; here it's: fix the
code, refold, atomically swap:

In [ ]:
buggy = SearchView(lowercase=False)
buggy.process(log)
assert "p7" not in buggy.find("coffee"), "the bug: 'Coffee' was indexed case-sensitively"

fixed = SearchView(lowercase=True)     # 1. fix the code
fixed.process(log)                     # 2. rebuild the ENTIRE view from the log
search = fixed                         # 3. atomic swap (a pointer flip / DNS change)
assert "p7" in search.find("coffee"), "rebuilt view: the post is findable"
print("bug fixed by refold-and-swap; nobody edited data by hand")

This is nb 10's "derived datasets are build artifacts" made operational, and it is the
strongest argument for keeping **raw, immutable history**: your future, smarter code can
always reinterpret it. (The **lambda architecture** ran a batch refold *and* a streaming
fold side by side, merging results; **kappa** — what we just did — observes that a
replayable log makes the separate batch layer redundant: same code, replayed.)

## 4. The price: asynchrony

Consumers run behind the log by their **lag** (nb 11). So the guarantee changes shape:

- **Integrity** (the log's order and content are the truth; every view *converges* to
  fold(log); nothing is lost or double-counted — given deterministic folds and
  effectively-once processing, nb 11 §2): kept, permanently.
- **Timeliness** (every view reflects this moment): explicitly given up. Views disagree
  *transiently* — and cross-view reads can observe it.

In [ ]:
log.append({"type": "user_upserted", "user_id": "u0", "name": "User 0, PhD"})

table.process(log)                     # the table applies the rename immediately...
assert table.users["u0"]["name"] == "User 0, PhD"
cache_view.process(log, upto=len(log) - 1)   # ...the cache is one event behind
assert cache_view.pages["u0"]["name"] == "User 0", \
    "cross-view read skew: table and cache disagree - RIGHT NOW"

cache_view.process(log)                # lag catches up...
assert cache_view.pages["u0"]["name"] == "User 0, PhD", "...and convergence is certain"
print("timeliness traded, integrity kept: stale for a moment, wrong never")

That transient disagreement is nb 05's replication lag reborn one level up — and the same
remedies apply (read-your-writes via the log offset your write got; monotonic reads via
sticky views). What the unbundled design *cannot* give you: an atomic transaction across
views (the log serializes writes, not multi-view reads), and synchronous constraints like
"reject the write unless the search index is updated too" — if you need those, that data
belongs *inside* one transactional store (nb 07). Unbundling is a trade you make per
dataset, not a religion.

One more honest note: views with **external side effects** (send a notification when
someone gets a follower) are not refoldable — replaying history would re-send six months of
notifications. Side-effecting consumers need idempotency keys and never participate in
rebuilds; keep them at the edges (the **end-to-end argument**: only the endpoints can make
the whole pipeline effectively-once — dedup in the middle helps performance, not
correctness).

## 5. Exercises

1. **Blue-green views.** Design the zero-downtime procedure to change the search view's
   tokenizer in production (the log keeps growing while you rebuild). When exactly do you
   flip readers, and what do you do with the old view?
2. **The right to be forgotten.** The log is immutable; a user invokes GDPR deletion.
   Reconcile the two using ideas you already own (nb 03's tombstones + compaction), and
   explain the crypto-shredding variant (per-user encryption keys) for logs you can't
   rewrite.
3. **Side effects vs state.** Classify these consumers as refoldable-state or
   side-effecting, and give each its safety mechanism: (a) likes counter, (b) welcome-email
   sender, (c) fraud-score table, (d) push-notification fanout.
4. **Cross-partition builds.** The real log is partitioned (nb 11): user events on the
   `users` partition, posts on `posts`. The profile-cache consumer may see a post before
   its author's user event. Give two coping designs and their costs.
5. **Why not 2PC?** Compare "log + async views" against "distributed transaction (2PC)
   spanning DB + search + cache on every write" on: write availability (nb 09's blocking),
   write latency (nb 01), adding a new store, and fixing a corrupted store.

### Solutions

**1.** Start view-v2 at offset 0 while v1 keeps serving; v2 churns through history and
then *stays* subscribed, trailing the head like any consumer. When v2's lag ≈ 0, flip the
read path (feature flag / router) — ideally after dark-reading v2 and diffing answers
against v1 for a while. Keep v1 consuming for the rollback window, then retire it. (This
is nb 06's live-resharding answer, and it works for *any* deterministic view — one
procedure to rule all migrations.)

**2.** Deletion becomes an *event*: append `user_erased(u)`; every view's fold treats it
as authoritative (drop u's data), and the log itself gets compacted (nb 03/nb 11:
log compaction keeps the latest record per key — a tombstone eventually erases priors).
For archives/backups you can't rewrite: encrypt each user's payloads with a per-user key
kept in a mutable store; deletion = destroy the key — the immutable bytes remain,
unreadable ("crypto-shredding"). Note the honest asymmetry: derived views purge on replay;
already-exported side effects don't un-happen.

**3.** (a) Refoldable state — needs only determinism (replay-safe). (b) Side-effecting —
idempotency key = user_id (nb 11 §2), excluded from rebuilds. (c) Refoldable state —
determinism; if the model changes, that's a rebuild (and exactly what nb 20 calls
retraining-as-refold). (d) Side-effecting — idempotency key = (user, event id) + a
freshness window (don't push about week-old events after downtime; timeliness matters to
humans).

**4.** (a) *Tolerate + repair*: the cache stores orphan posts under the user_id and
completes the page when the user event arrives (our `setdefault` calls already do this) —
cheap, briefly incomplete pages. (b) *Co-partition by user_id*: route both event types by
the same key (nb 11 exercise 1) so one partition orders them — removes the race for
per-user views, at the cost of key design and hot-user skew (nb 06). A third heavyweight
option — buffering until dependencies arrive — reinvents multi-stream watermarks.

**5.** 2PC: every write pays the slowest store's latency and *blocks* if any participant
or the coordinator stalls (nb 09) — the cache's hiccup becomes the checkout's outage;
adding a store means joining the transaction (touch every writer); a corrupted index needs
a bespoke repair. Log+async: writes commit at log speed regardless of view health; a new
store is a new consumer at offset 0; corruption is refold-and-swap. You pay with
timeliness and with designing for staleness. For *most* cross-store consistency, async
wins; reserve transactions for invariants that cannot be eventually (uniqueness, money —
nb 07/09).

## 6. Takeaways — and Part 1, folded up

- Dual writes to N stores cannot be consistent: no single order exists. **Create the
  order**: one append-only log as the system of record; everything else is a
  deterministic fold — table, index, cache, analytics.
- Derived views buy: anomaly-free denormalization (nb 02's trap, defused), instant new
  views with full history, bug repair by rebuild (human fault tolerance), per-view
  engines (nb 03's zoo, justified).
- The price: **timeliness**. Integrity is kept by ordered logs + effectively-once
  processing (nb 11); staleness is managed by nb 05's guarantees, per feature. Side
  effects live at the edges with idempotency keys (end-to-end argument).
- **Part 1 in one sentence**: keep an immutable ordered history of facts; derive every
  convenient shape from it; coordinate (nb 07/09) only where invariants truly demand it —
  and know the price of each guarantee you claim.

**Further reading.** DDIA ch. 12; Kleppmann, *Turning the database inside-out* (2015
talk) — this notebook's thesis; Helland, *Immutability Changes Everything* (CIDR 2015).

**Next:** Part 2. [13 · ML systems overview](../part2-ml-systems/13-ml-systems-overview.ipynb)
— everything so far assumed you knew what to compute. Now the computation is *learned from
data*, and a new set of failure modes arrives.